# LeetCode 459 — Repeated Substring Pattern bằng Python

**Link đề:** https://leetcode.com/problems/repeated-substring-pattern/
**Độ khó:** Easy · **Chủ đề:** String, String Matching, KMP (prefix function)

```python
class Solution:
    def repeatedSubstringPattern(self, s: str) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu đề: chuỗi `s` có được tạo bằng cách lặp **một chuỗi con ≥ 2 lần** không?
2. Ôn slicing `s[a:b]`, phép nhân chuỗi `"ab" * 3`, `%` để kiểm tra chia hết, toán tử `in` với chuỗi.
3. Học 4 lời giải: thử mọi độ dài ước → so sánh dịch chuyển `s[i] == s[i-L]` → **mẹo nhân đôi chuỗi** `s in (s+s)[1:-1]` → **KMP prefix function**.
4. Trace, chứng minh tính đúng của mẹo nhân đôi và công thức KMP, phân tích Big-O.
5. Nhận dạng pattern "chu kỳ / period của chuỗi", làm 5 bài tập có gợi ý và đáp án ở cuối.

**Cách học:** đọc markdown → đoán output cell kế tiếp → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho chuỗi `s`. Kiểm tra xem `s` có thể được tạo ra bằng cách lấy **một chuỗi con** của nó rồi **nối nhiều bản sao** (ít nhất 2) lại với nhau hay không.

| `s` | Output | Giải thích |
|---|:---:|---|
| `"abab"` | `True` | `"ab"` × 2 |
| `"aba"` | `False` | không có khối nào lặp đều |
| `"abcabcabcabc"` | `True` | `"abc"` × 4 (hoặc `"abcabc"` × 2) |
| `"a"` | `False` | cần ít nhất 2 bản sao |
| `"aaaa"` | `True` | `"a"` × 4 |

**Constraints:** `1 <= s.length <= 10^4`, `s` chỉ gồm chữ thường tiếng Anh.

Lưu ý: "chuỗi con" ở đây phải là **tiền tố** (prefix) của `s` — vì `s` bắt đầu bằng bản sao đầu tiên của khối lặp.

In [1]:
s = "abcabcabcabc"
block = s[:3]
print("khối:", block, "| lặp 4 lần:", block * 4, "| bằng s?", block * 4 == s)

khối: abc | lặp 4 lần: abcabcabcabc | bằng s? True


Câu hỏi đặt ra trước khi code:
- Khối lặp có thể dài bao nhiêu? Có cần thử mọi độ dài không?
- Nếu độ dài khối là `L` thì `L` có quan hệ gì với `n = len(s)`?
- Có mẹo nào kiểm tra "chuỗi tuần hoàn" chỉ bằng một phép tìm kiếm không?

## 2. Nền tảng Python cần biết

### 2.1. Slicing `s[a:b]` — cắt chuỗi

`s[a:b]` lấy ký tự từ chỉ số `a` đến **trước** `b`. Bỏ `a` nghĩa là từ đầu, bỏ `b` là đến hết. Chỉ số âm đếm từ cuối: `s[-1]` là ký tự cuối, `s[1:-1]` bỏ ký tự đầu và cuối.

In [2]:
s = "abcdef"
print(s[:2])     # 'ab'   -> tiền tố dài 2
print(s[2:5])    # 'cde'
print(s[3:])     # 'def'  -> từ chỉ số 3 đến hết
print(s[1:-1])   # 'bcde' -> bỏ ký tự đầu và cuối
print(s[-1])     # 'f'

ab
cde
def
bcde
f


### 2.2. Nhân chuỗi và nối chuỗi

- `"ab" * 3` → `"ababab"`: lặp chuỗi 3 lần.
- `"ab" + "cd"` → `"abcd"`: nối chuỗi.

Chi phí: tạo chuỗi mới dài `k` tốn `O(k)` thời gian và bộ nhớ.

In [3]:
print("ab" * 3)
print("xyz" + "xyz")
print("a" * 0 == "")    # nhân 0 lần ra chuỗi rỗng

ababab
xyzxyz
True


### 2.3. Phép `%` để kiểm tra chia hết

`n % L == 0` nghĩa là `L` là **ước** của `n`. Khối lặp dài `L` chỉ có thể ghép đúng thành chuỗi dài `n` khi `L` chia hết `n` (vì `n = L × số_lần_lặp`).

In [4]:
n = 12
print("Các ước của 12 (trừ chính nó):", [L for L in range(1, n) if n % L == 0])
print("n = 7 ->", [L for L in range(1, 7) if 7 % L == 0])   # số nguyên tố: chỉ có 1

Các ước của 12 (trừ chính nó): [1, 2, 3, 4, 6]
n = 7 -> [1]


### 2.4. Toán tử `in` với chuỗi — tìm chuỗi con

`sub in text` trả `True` nếu `sub` xuất hiện **liền mạch** trong `text`. `text.find(sub, start)` trả vị trí đầu tiên tìm thấy (từ `start`), hoặc `-1` nếu không có.

In [5]:
text = "abababab"
print("bab" in text)            # True
print("bb" in text)             # False
print(text.find("abab"))        # 0
print(text.find("abab", 1))     # 2: lần xuất hiện đầu tiên bắt đầu từ chỉ số >= 1

True
False
0
2


## 3. Cách 1 — Thử mọi độ dài khối là ước của `n`

Trực giác: thử từng `L = 1, 2, ..., n // 2` (khối phải lặp ≥ 2 lần nên `L <= n/2`). Nếu `n % L == 0` và `s[:L] * (n // L) == s` thì trả `True`.

Số ước của `n` rất nhỏ (với `n <= 10^4` tối đa 64 ước), mỗi lần so sánh tốn `O(n)` → tổng `O(n · d(n))`, đủ nhanh cho đề này.

In [6]:
def repeated_divisors(s: str) -> bool:
    n = len(s)
    for L in range(1, n // 2 + 1):
        if n % L == 0 and s[:L] * (n // L) == s:
            return True
    return False

for t in ["abab", "aba", "abcabcabcabc", "a", "aaaa"]:
    print(f"{t!r:16} -> {repeated_divisors(t)}")

'abab'           -> True
'aba'            -> False
'abcabcabcabc'   -> True
'a'              -> False
'aaaa'           -> True


Giải thích `{t!r:16}` trong f-string: `!r` in theo dạng `repr` (có dấu nháy), `:16` căn lề trái độ rộng 16 ký tự cho bảng thẳng hàng.

### 3.1. Trace cách thử ước

Với `s = "abcabcabcabc"` (n = 12): thử `L = 1` (`"a"*12` ≠ s), `L = 2` (`"ab"*6` ≠ s), `L = 3` (`"abc"*4` = s) → `True`. Các `L` không chia hết `n` (5, 7, ...) bị bỏ qua ngay.

In [7]:
def trace_divisors(s):
    n = len(s)
    for L in range(1, n // 2 + 1):
        if n % L != 0:
            print(f"L={L}: {n} % {L} != 0 -> bỏ qua")
            continue
        candidate = s[:L] * (n // L)
        ok = candidate == s
        print(f"L={L}: khối {s[:L]!r} x {n // L} = {candidate!r} -> {'KHỚP' if ok else 'khác'}")
        if ok:
            return True
    print("Không có L nào khớp")
    return False

trace_divisors("abcabcabcabc")
print()
trace_divisors("abaaba")   # n = 6

L=1: khối 'a' x 12 = 'aaaaaaaaaaaa' -> khác
L=2: khối 'ab' x 6 = 'abababababab' -> khác
L=3: khối 'abc' x 4 = 'abcabcabcabc' -> KHỚP

L=1: khối 'a' x 6 = 'aaaaaa' -> khác
L=2: khối 'ab' x 3 = 'ababab' -> khác
L=3: khối 'aba' x 2 = 'abaaba' -> KHỚP


True

### 3.2. Cách 1b — so sánh dịch chuyển, không tạo chuỗi mới

Chuỗi có chu kỳ `L` ⇔ `s[i] == s[i - L]` với mọi `i >= L`. Kiểm tra trực tiếp như vậy tránh tạo chuỗi `s[:L] * k` (tiết kiệm bộ nhớ, thoát sớm khi gặp ký tự khác). `all(...)` trả `True` nếu mọi phần tử của generator đều đúng, và dừng ngay ở phần tử sai đầu tiên.

In [8]:
print(all(x > 0 for x in [1, 2, 3]), all(x > 0 for x in [1, -2, 3]))

def repeated_shift_compare(s: str) -> bool:
    n = len(s)
    for L in range(1, n // 2 + 1):
        if n % L == 0 and all(s[i] == s[i - L] for i in range(L, n)):
            return True
    return False

print([repeated_shift_compare(t) for t in ["abab", "aba", "abcabcabcabc", "a"]])

True False
[True, False, True, False]


## 4. Bước đột phá — mẹo nhân đôi chuỗi

Ghép `t = s + s`, rồi bỏ ký tự đầu và cuối: `t[1:-1]`. Khẳng định:

> `s` là chuỗi lặp ⇔ `s` xuất hiện trong `(s + s)[1:-1]`.

Trực giác bằng **phép xoay** (rotation): mọi chuỗi con dài `n` của `s + s` bắt đầu ở vị trí `k` (0 ≤ k < n) chính là `s` **xoay trái** `k` ký tự. Bỏ ký tự đầu và cuối nghĩa là ta loại 2 vị trí `k = 0` và `k = n` (cả hai đều cho đúng `s` một cách "tầm thường"). Vậy câu hỏi trở thành: **có phép xoay không tầm thường nào biến `s` thành chính nó không?**

- `"abab"` xoay 2 → `"abab"` ✔ → lặp.
- `"aba"` xoay 1 → `"baa"`, xoay 2 → `"aab"` ✘ → không lặp.

In [9]:
def rotations(s):
    return [s[k:] + s[:k] for k in range(len(s))]

for s in ["abab", "aba"]:
    print(s, "-> các phép xoay:", rotations(s))
    t = (s + s)[1:-1]
    print("   (s+s)[1:-1] =", t, "| s in t ?", s in t)

abab -> các phép xoay: ['abab', 'baba', 'abab', 'baba']
   (s+s)[1:-1] = bababa | s in t ? True
aba -> các phép xoay: ['aba', 'baa', 'aab']
   (s+s)[1:-1] = baab | s in t ? False


### 4.1. Vì sao "xoay `k` ra chính nó" ⇔ "lặp"?

- **(⇐)** Nếu `s = p * m` với `|p| = L`, `m >= 2` thì xoay `L` ký tự cho lại `s`, và `0 < L < n`.
- **(⇒)** Nếu xoay `k` (0 < k < n) cho lại `s` thì `s[i] = s[(i + k) mod n]` với mọi `i`. Đặt `L = gcd(k, n)`; lặp lại phép xoay `k` nhiều lần ta được `s` bất biến khi xoay `L` (vì có các số nguyên `a, b` với `a·k + b·n = L`). Vậy `s[i] = s[i + L]`, tức `s` lặp khối dài `L`, và `L` chia hết `n`, `L <= k < n` → ít nhất 2 bản sao.

Ta có thể kiểm chứng: vị trí đầu tiên `s` xuất hiện trong `s + s` kể từ chỉ số 1 chính là **độ dài khối lặp ngắn nhất** (nếu nhỏ hơn `n`).

In [10]:
for s in ["abcabcabcabc", "aaaa", "abab", "aba", "abcab"]:
    k = (s + s).find(s, 1)
    print(f"{s!r:16} xuất hiện lại ở vị trí {k:>2} -> {'khối ' + repr(s[:k]) if k < len(s) else 'không lặp'}")

'abcabcabcabc'   xuất hiện lại ở vị trí  3 -> khối 'abc'
'aaaa'           xuất hiện lại ở vị trí  1 -> khối 'a'
'abab'           xuất hiện lại ở vị trí  2 -> khối 'ab'
'aba'            xuất hiện lại ở vị trí  3 -> không lặp
'abcab'          xuất hiện lại ở vị trí  5 -> không lặp


## 5. Lời giải chính — mẹo nhân đôi (mổ xẻ rồi ghép)

```python
def repeated(s):
    doubled = s + s          # bước 1: dài 2n, chứa mọi phép xoay của s
    inner = doubled[1:-1]    # bước 2: bỏ 2 phép xoay tầm thường (k = 0 và k = n)
    return s in inner        # bước 3: còn phép xoay nào trùng s không?
```

Ba dòng, nhưng mỗi dòng có lý do toán học rõ ràng ở mục 4.

In [11]:
def repeated_double(s: str) -> bool:
    return s in (s + s)[1:-1]

for t in ["abab", "aba", "abcabcabcabc", "a", "aaaa", "ab"]:
    print(f"{t!r:16} -> {repeated_double(t)}")

'abab'           -> True
'aba'            -> False
'abcabcabcabc'   -> True
'a'              -> False
'aaaa'           -> True
'ab'             -> False


Ca biên `s = "a"`: `s + s = "aa"`, `[1:-1]` = `""`, `"a" in ""` là `False` → đúng (một ký tự không thể lặp ≥ 2 lần).

## 6. TRACE — xem `s` trượt trên `(s+s)[1:-1]`

Bảng với `s = "abab"`, `inner = "bababa"` (dài 6). Thử đặt `s` tại mọi vị trí `p` trong `inner`:

| p | `inner[p:p+4]` | bằng `s`? | tương ứng xoay `k = p+1` |
|---|---|---|---|
| 0 | `"baba"` | Không | 1 |
| 1 | `"abab"` | **Có** | 2 |
| 2 | `"baba"` | Không | 3 |

Dừng ở `p = 1` → `True`, khối lặp dài `k = 2`.

In [12]:
def trace_double(s):
    inner = (s + s)[1:-1]
    n = len(s)
    print(f"s = {s!r}, inner = {inner!r}")
    for p in range(len(inner) - n + 1):
        window = inner[p:p + n]
        hit = window == s
        print(f"  p={p}: {window!r} {'== s -> True (xoay ' + str(p + 1) + ')' if hit else '!= s'}")
        if hit:
            return True
    print("  không vị trí nào khớp -> False")
    return False

trace_double("abab")
trace_double("abcab")

s = 'abab', inner = 'bababa'
  p=0: 'baba' != s
  p=1: 'abab' == s -> True (xoay 2)
s = 'abcab', inner = 'bcababca'
  p=0: 'bcaba' != s
  p=1: 'cabab' != s
  p=2: 'ababc' != s
  p=3: 'babca' != s
  không vị trí nào khớp -> False


False

## 7. Cách 3 — KMP prefix function (độ dài `O(n)` chắc chắn)

`s in t` của Python rất nhanh trong thực tế, nhưng để có bảo đảm lý thuyết `O(n)` ta dùng **prefix function** (mảng `lps` — longest proper prefix which is also suffix) của thuật toán KMP.

`lps[i]` = độ dài **tiền tố thật sự dài nhất** của `s[:i+1]` đồng thời là **hậu tố** của nó ("thật sự" = khác chính nó).

Ví dụ `s = "abcabcabc"`: `lps = [0,0,0,1,2,3,4,5,6]`. `lps[-1] = 6`: tiền tố `"abcabc"` cũng là hậu tố.

**Công thức:** đặt `p = n - lps[-1]` (độ dài **chu kỳ ngắn nhất**). Khi đó
> `s` lặp ⇔ `lps[-1] > 0` **và** `n % p == 0`.

In [13]:
def prefix_function(s: str) -> list[int]:
    lps = [0] * len(s)
    k = 0                         # độ dài tiền tố-hậu tố hiện tại
    for i in range(1, len(s)):
        while k > 0 and s[i] != s[k]:
            k = lps[k - 1]        # lùi về tiền tố-hậu tố ngắn hơn
        if s[i] == s[k]:
            k += 1
        lps[i] = k
    return lps

for t in ["abcabcabc", "abab", "aba", "aabaab", "abac"]:
    print(f"{t!r:12} lps = {prefix_function(t)}")

'abcabcabc'  lps = [0, 0, 0, 1, 2, 3, 4, 5, 6]
'abab'       lps = [0, 0, 1, 2]
'aba'        lps = [0, 0, 1]
'aabaab'     lps = [0, 1, 0, 1, 2, 3]
'abac'       lps = [0, 0, 1, 0]


### 7.1. Trace cách xây `lps` cho `"aabaab"`

| i | `s[i]` | `k` trước | so `s[i]` với `s[k]` | `k` sau | `lps[i]` |
|---|---|---|---|---|---|
| 1 | a | 0 | a == a | 1 | 1 |
| 2 | b | 1 | b != a → lùi `k = lps[0] = 0`; b != a | 0 | 0 |
| 3 | a | 0 | a == a | 1 | 1 |
| 4 | a | 1 | a == a | 2 | 2 |
| 5 | b | 2 | b == b | 3 | 3 |

`lps[-1] = 3`, `n = 6`, `p = 3`, `6 % 3 == 0` → lặp khối `"aab"`.

In [14]:
def trace_prefix_function(s):
    lps = [0] * len(s)
    k = 0
    for i in range(1, len(s)):
        k_before = k
        while k > 0 and s[i] != s[k]:
            print(f"  i={i}: s[i]={s[i]!r} != s[{k}]={s[k]!r} -> lùi k = lps[{k-1}] = {lps[k-1]}")
            k = lps[k - 1]
        if s[i] == s[k]:
            k += 1
        lps[i] = k
        print(f"i={i}, s[i]={s[i]!r}, k: {k_before} -> {k}, lps = {lps}")
    n = len(s)
    p = n - lps[-1]
    print(f"lps[-1]={lps[-1]}, chu kỳ p = {p}, n % p = {n % p} -> {lps[-1] > 0 and n % p == 0}")

trace_prefix_function("aabaab")

i=1, s[i]='a', k: 0 -> 1, lps = [0, 1, 0, 0, 0, 0]
  i=2: s[i]='b' != s[1]='a' -> lùi k = lps[0] = 0
i=2, s[i]='b', k: 1 -> 0, lps = [0, 1, 0, 0, 0, 0]
i=3, s[i]='a', k: 0 -> 1, lps = [0, 1, 0, 1, 0, 0]
i=4, s[i]='a', k: 1 -> 2, lps = [0, 1, 0, 1, 2, 0]
i=5, s[i]='b', k: 2 -> 3, lps = [0, 1, 0, 1, 2, 3]
lps[-1]=3, chu kỳ p = 3, n % p = 0 -> True


In [15]:
def repeated_kmp(s: str) -> bool:
    lps = prefix_function(s)
    n = len(s)
    longest = lps[-1]
    return longest > 0 and n % (n - longest) == 0

for t in ["abab", "aba", "abcabcabcabc", "a", "aaaa", "abac", "abaaba"]:
    print(f"{t!r:16} -> {repeated_kmp(t)}")

'abab'           -> True
'aba'            -> False
'abcabcabcabc'   -> True
'a'              -> False
'aaaa'           -> True
'abac'           -> False
'abaaba'         -> True


### 7.2. Vì sao công thức KMP đúng?

- Nếu tiền tố dài `lps[-1]` trùng hậu tố dài `lps[-1]`, thì `s[i] = s[i + p]` với mọi `i` hợp lệ, `p = n - lps[-1]` → `s` có **chu kỳ** `p` (có thể khối cuối bị cụt).
- `p` là chu kỳ **nhỏ nhất** vì `lps[-1]` là lớn nhất.
- Khối cuối không bị cụt ⇔ `p` chia hết `n` ⇔ `s = s[:p] * (n // p)`. Cần `lps[-1] > 0` để `p < n` (tức ≥ 2 bản sao).
- Nếu `n % p != 0` thì không chu kỳ nào lớn hơn chia hết `n` cũng không đủ (một định lý về chu kỳ — Fine–Wilf); ta kiểm chứng thực nghiệm ở mục test.

Ví dụ `"abaaba"`: `lps[-1] = 3`, `p = 3`, `6 % 3 == 0` → `"aba" * 2` ✔. Còn `"abcab"`: `lps[-1] = 2`, `p = 3`, `5 % 3 != 0` → không lặp.

## 8. Code nộp LeetCode (Python 3)

Bản chính: mẹo nhân đôi (ngắn, nhanh trong thực tế).

In [16]:
class Solution:
    def repeatedSubstringPattern(self, s: str) -> bool:
        return s in (s + s)[1:-1]

sol = Solution()
print(sol.repeatedSubstringPattern("abab"))           # True
print(sol.repeatedSubstringPattern("aba"))            # False
print(sol.repeatedSubstringPattern("abcabcabcabc"))   # True

True
False
True


Bản KMP (bảo đảm `O(n)` tuyệt đối) cũng Accepted:

```python
class Solution:
    def repeatedSubstringPattern(self, s: str) -> bool:
        lps = [0] * len(s)
        k = 0
        for i in range(1, len(s)):
            while k > 0 and s[i] != s[k]:
                k = lps[k - 1]
            if s[i] == s[k]:
                k += 1
            lps[i] = k
        n = len(s)
        return lps[-1] > 0 and n % (n - lps[-1]) == 0
```

## 9. Vì sao đúng? (tổng hợp invariant)

- **Cách thử ước:** nếu `s = p * m` thì `|p|` là ước của `n` và `|p| <= n/2` → vòng lặp chắc chắn thử tới `L = |p|` và khớp. Ngược lại, khi trả `True` ta đã thấy tận mắt `s[:L] * (n//L) == s` với `n//L >= 2`.
- **Nhân đôi:** `(s+s)[1:-1]` chứa đúng các phép xoay `k = 1..n-1`; `s` lặp ⇔ có phép xoay không tầm thường cố định `s` (chứng minh ở 4.1).
- **KMP:** invariant của vòng lặp: sau bước `i`, `k = lps[i]` là độ dài tiền tố-hậu tố dài nhất của `s[:i+1]`. Bước lùi `k = lps[k-1]` thử ứng viên dài nhất kế tiếp nên không bỏ sót.

## 10. Phân tích Time / Space

Gọi `n = len(s)`, `d(n)` = số ước của `n` (với `n <= 10^4`, `d(n) <= 64`).

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. Thử ước + nhân chuỗi | `O(n · d(n))` | `O(n)` | dễ nghĩ nhất |
| 1b. Thử ước + so dịch chuyển | `O(n · d(n))` | `O(1)` | thoát sớm |
| 2. Nhân đôi `s in (s+s)[1:-1]` | `O(n)` thực tế* | `O(n)` | ngắn nhất |
| 3. KMP prefix function | `O(n)` | `O(n)` | bảo đảm lý thuyết |

\* Tìm chuỗi con của CPython dùng thuật toán kết hợp (two-way / Crochemore–Perrin cho mẫu dài từ Python 3.10), chạy tuyến tính trong thực tế; tránh khẳng định tuyệt đối cho mọi phiên bản.

Vì sao vòng `while` trong KMP vẫn tổng `O(n)`? `k` tăng tối đa 1 mỗi vòng `for` (tổng ≤ n), và mỗi lần lùi giảm `k` ít nhất 1 → tổng số lần lùi ≤ tổng số lần tăng ≤ n.

## 11. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

cases = [
    ("đề 1", "abab", True),
    ("đề 2", "aba", False),
    ("đề 3", "abcabcabcabc", True),
    ("một ký tự", "a", False),
    ("hai ký tự giống", "aa", True),
    ("hai ký tự khác", "ab", False),
    ("toàn một chữ", "zzzzzzz", True),
    ("n nguyên tố, không đều", "aabaa", False),
    ("chu kỳ không chia hết n", "abcab", False),
    ("palindrome lặp", "abaaba", True),
    ("gần lặp, sai ký tự cuối", "abcabcabd", False),
]
sol = Solution()
for name, s, expected in cases:
    check(name, sol.repeatedSubstringPattern(s), expected)

PASS  đề 1: True
PASS  đề 2: False
PASS  đề 3: True
PASS  một ký tự: False
PASS  hai ký tự giống: True
PASS  hai ký tự khác: False
PASS  toàn một chữ: True
PASS  n nguyên tố, không đều: False
PASS  chu kỳ không chia hết n: False
PASS  palindrome lặp: True
PASS  gần lặp, sai ký tự cuối: False


In [18]:
import random
random.seed(459)
methods = [repeated_divisors, repeated_shift_compare, repeated_double, repeated_kmp]
count_true = 0
for _ in range(3000):
    if random.random() < 0.5:
        block = "".join(random.choice("ab") for _ in range(random.randint(1, 4)))
        s = block * random.randint(1, 4)       # thường là chuỗi lặp
    else:
        s = "".join(random.choice("ab") for _ in range(random.randint(1, 10)))
    results = [f(s) for f in methods]
    assert len(set(results)) == 1, (s, results)
    count_true += results[0]
print("3000 chuỗi ngẫu nhiên: 4 cách khớp nhau; số chuỗi lặp =", count_true)

3000 chuỗi ngẫu nhiên: 4 cách khớp nhau; số chuỗi lặp = 1418


In [19]:
from itertools import product
# Vét cạn tất cả chuỗi trên bảng chữ {a, b} dài 1..12 (8190 chuỗi)
total = 0
for n in range(1, 13):
    for tup in product("ab", repeat=n):
        s = "".join(tup)
        assert repeated_divisors(s) == repeated_double(s) == repeated_kmp(s), s
        total += 1
print("Đã kiểm vét cạn", total, "chuỗi: các cách khớp nhau")

Đã kiểm vét cạn 8190 chuỗi: các cách khớp nhau


Giải thích `itertools.product("ab", repeat=n)`: sinh **mọi bộ** độ dài `n` lấy ký tự từ `"ab"` (tích Descartes), ví dụ `n=2` → `('a','a'), ('a','b'), ('b','a'), ('b','b')`. Đây là cách sinh test vét cạn rất tiện.

## 12. Lỗi người mới hay mắc

### Lỗi 1: cho `L` chạy tới `n` (gồm chính `s`)
Khi `L = n`, `s[:n] * 1 == s` luôn đúng → mọi chuỗi đều bị báo `True`. Khối phải lặp **ít nhất 2 lần**, nên `L <= n // 2`.

In [20]:
def wrong_include_n(s):
    n = len(s)
    for L in range(1, n + 1):           # SAI: gồm L = n
        if n % L == 0 and s[:L] * (n // L) == s:
            return True
    return False

print("sai :", wrong_include_n("aba"))
print("đúng:", repeated_divisors("aba"))

sai : True
đúng: False


### Lỗi 2: chỉ kiểm tra `s in s + s` (quên bỏ đầu/cuối)
`s` luôn nằm trong `s + s` ở vị trí 0 → luôn `True`.

In [21]:
print("sai :", "aba" in "aba" + "aba")
print("đúng:", "aba" in ("aba" + "aba")[1:-1])

sai : True
đúng: False


### Lỗi 3: quên kiểm tra `n % L == 0`
Không kiểm tra chia hết (và cho `L` chạy tới `n - 1`), nếu so sánh bằng dịch chuyển `s[i] == s[i-L]` thì `"abcab"` với `L = 3` vẫn thỏa (chu kỳ 3) dù khối cuối bị cụt → báo `True` sai.

In [22]:
def wrong_no_divisible(s):
    n = len(s)
    for L in range(1, n):
        if all(s[i] == s[i - L] for i in range(L, n)):   # SAI: thiếu n % L == 0
            return True
    return False

print("sai :", wrong_no_divisible("abcab"))
print("đúng:", repeated_shift_compare("abcab"))

sai : True
đúng: False


### Lỗi 4: KMP quên điều kiện `lps[-1] > 0`
Nếu `lps[-1] = 0` thì `p = n`, `n % n == 0` → báo `True` sai cho mọi chuỗi không có tiền tố-hậu tố chung (ví dụ `"ab"`).

### Lỗi 5: chỉ thử khối dài 1 hoặc chỉ thử `n // 2`
Chỉ thử `L = n // 2` bỏ sót `"abcabcabc"` (n = 9, khối 3 lặp 3 lần, `n // 2 = 4` không chia hết 9).

In [23]:
def wrong_kmp_no_zero_check(s):
    lps = prefix_function(s)
    n = len(s)
    return n % (n - lps[-1]) == 0

def wrong_half_only(s):
    n = len(s)
    return n % 2 == 0 and s[: n // 2] * 2 == s

print("KMP thiếu lps>0 với 'ab':", wrong_kmp_no_zero_check("ab"), "| đúng:", repeated_kmp("ab"))
print("chỉ thử n//2 với 'abcabcabc':", wrong_half_only("abcabcabc"), "| đúng:", repeated_double("abcabcabc"))

KMP thiếu lps>0 với 'ab': True | đúng: False
chỉ thử n//2 với 'abcabcabc': False | đúng: True


## 13. Mẹo nhận dạng pattern

- "Chuỗi có phải lặp lại một khối?" / "chu kỳ nhỏ nhất của chuỗi?" → **prefix function**: chu kỳ = `n - lps[-1]`.
- "Chuỗi `B` có phải phép xoay của `A`?" → `len(A) == len(B) and B in A + A` (LeetCode 796 Rotate String).
- Độ dài khối luôn là **ước** của `n` → chỉ cần thử ước.

Bài cùng họ: 796 Rotate String, 28 Find the Index of the First Occurrence (KMP), 686 Repeated String Match, 1392 Longest Happy Prefix (chính là `lps[-1]`), 214 Shortest Palindrome.

## 14. Biến thể

### 14.1. Trả về **khối lặp ngắn nhất** (hoặc chính `s` nếu không lặp)

In [24]:
def shortest_block(s: str) -> str:
    k = (s + s).find(s, 1)   # luôn tìm thấy, muộn nhất ở vị trí n
    return s[:k]

for t in ["abcabcabcabc", "aaaa", "aba", "abaaba"]:
    print(f"{t!r:16} -> khối ngắn nhất {shortest_block(t)!r}, số lần lặp {len(t) // len(shortest_block(t))}")

'abcabcabcabc'   -> khối ngắn nhất 'abc', số lần lặp 4
'aaaa'           -> khối ngắn nhất 'a', số lần lặp 4
'aba'            -> khối ngắn nhất 'aba', số lần lặp 1
'abaaba'         -> khối ngắn nhất 'aba', số lần lặp 2


### 14.2. Rotate String (LeetCode 796)

In [25]:
def rotate_string(a: str, b: str) -> bool:
    return len(a) == len(b) and b in a + a

print(rotate_string("abcde", "cdeab"), rotate_string("abcde", "abced"))

True False


## 15. Bài tập tự luyện (tăng dần)

> Viết code vào các hàm TODO; test để ở dạng comment. Đáp án ở mục 17.

### Bài 1 — Dự đoán output
Không chạy code, cho biết `("xyzxyz" + "xyzxyz")[1:-1]` là chuỗi gì và `"xyzxyz"` xuất hiện ở vị trí nào trong đó (dùng `find`).

### Bài 2 — Tính tay prefix function
Lập bảng `lps` cho `s = "ababab"`. Từ đó chu kỳ `p` bằng bao nhiêu? Có lặp không?

### Bài 3 — Hoàn thiện cách thử ước

In [26]:
def ex3_repeated(s: str) -> bool:
    # TODO 1: n = len(s)
    # TODO 2: duyệt L từ 1 đến n // 2
    # TODO 3: nếu n % L == 0 và s[:L] * (n // L) == s -> True
    pass

# assert ex3_repeated("abab") is True
# assert ex3_repeated("aba") is False

### Bài 4 — Đếm số cách viết `s` thành khối lặp
Trả về số độ dài `L` (`1 <= L < n`) sao cho `s` là khối dài `L` lặp lại. Ví dụ `"abababab"` → `2` (L = 2 và L = 4); `"aaaa"` → `2` (L = 1, 2).

In [27]:
def ex4_count_block_lengths(s: str) -> int:
    # TODO
    pass

# assert ex4_count_block_lengths("abababab") == 2
# assert ex4_count_block_lengths("aaaa") == 2
# assert ex4_count_block_lengths("abc") == 0

### Bài 5 — Mini challenge: Repeated String Match (LeetCode 686)
Trả về **số lần ít nhất** phải lặp chuỗi `a` để `b` trở thành chuỗi con; nếu không thể, trả `-1`. Ví dụ `a = "abcd"`, `b = "cdabcdab"` → `3`.

In [28]:
def ex5_repeated_string_match(a: str, b: str) -> int:
    # TODO: thử số lần lặp q nhỏ nhất sao cho len(a*q) >= len(b), rồi q và q+1
    pass

# assert ex5_repeated_string_match("abcd", "cdabcdab") == 3
# assert ex5_repeated_string_match("a", "aa") == 2
# assert ex5_repeated_string_match("abc", "wxyz") == -1

## 16. Gợi ý 3 mức

**Bài 1** — Mức 1: `s + s` dài 12. Mức 2: bỏ ký tự đầu `x` và cuối `z`. Mức 3: `"yzxyzxyzxy"`, tìm `"xyzxyz"` ở vị trí 2.

**Bài 2** — Mức 1: `lps[0] = 0` luôn. Mức 2: từ `i = 2` trở đi mỗi ký tự khớp tiếp → `k` tăng 1. Mức 3: `[0,0,1,2,3,4]`, `p = 6 - 4 = 2`.

**Bài 3** — Mức 1: chỉ thử `L` chia hết `n`. Mức 2: `range(1, n // 2 + 1)`. Mức 3: `return False` sau vòng lặp.

**Bài 4** — Mức 1: đếm thay vì `return True`. Mức 2: dùng lại điều kiện bài 3 cho mọi `L < n`. Mức 3: `sum(1 for L in range(1, n) if n % L == 0 and s[:L] * (n // L) == s)`.

**Bài 5** — Mức 1: `q = ceil(len(b) / len(a))` (dùng `-(-x // y)`). Mức 2: chuỗi con `b` có thể bắt đầu giữa một bản sao `a`, nên thử thêm `q + 1`. Mức 3: nếu `b in a * q` → `q`; elif `b in a * (q + 1)` → `q + 1`; else `-1`.

## 17. Đáp án tham khảo

**Bài 1:** `"yzxyzxyzxy"`, `find` trả `2` → `"xyzxyz"` lặp khối dài `2 + 1 = 3` (`"xyz"`).

**Bài 2:** `lps = [0, 0, 1, 2, 3, 4]`, `p = 2`, `6 % 2 == 0` → lặp khối `"ab"`.

In [29]:
def ans3_repeated(s):
    n = len(s)
    for L in range(1, n // 2 + 1):
        if n % L == 0 and s[:L] * (n // L) == s:
            return True
    return False

def ans4_count_block_lengths(s):
    n = len(s)
    return sum(1 for L in range(1, n) if n % L == 0 and s[:L] * (n // L) == s)

def ans5_repeated_string_match(a, b):
    q = -(-len(b) // len(a))          # làm tròn lên
    if b in a * q:
        return q
    if b in a * (q + 1):
        return q + 1
    return -1

assert ("xyzxyz" + "xyzxyz")[1:-1] == "yzxyzxyzxy"
assert ("xyzxyz" + "xyzxyz")[1:-1].find("xyzxyz") == 2
assert prefix_function("ababab") == [0, 0, 1, 2, 3, 4]
assert ans3_repeated("abab") is True and ans3_repeated("aba") is False
assert ans4_count_block_lengths("abababab") == 2
assert ans4_count_block_lengths("aaaa") == 2
assert ans4_count_block_lengths("abc") == 0
assert ans5_repeated_string_match("abcd", "cdabcdab") == 3
assert ans5_repeated_string_match("a", "aa") == 2
assert ans5_repeated_string_match("abc", "wxyz") == -1
print("Đáp án bài 1–5: PASS")

Đáp án bài 1–5: PASS


Giải thích `-(-x // y)`: `//` làm tròn **xuống** (về phía âm vô cùng). Đổi dấu hai lần biến nó thành làm tròn **lên** mà không cần `import math`. Ví dụ `-(-8 // 4) = 2`, `-(-9 // 4) = 3`.

In [30]:
import math
for x, y in [(8, 4), (9, 4), (1, 5)]:
    print(x, y, "->", -(-x // y), "| math.ceil:", math.ceil(x / y))

8 4 -> 2 | math.ceil: 2
9 4 -> 3 | math.ceil: 3
1 5 -> 1 | math.ceil: 1


## 18. Template nhớ nhanh + checklist submit

```python
# Chuỗi lặp khối?
return s in (s + s)[1:-1]

# Chu kỳ nhỏ nhất bằng prefix function
p = n - lps[-1]
is_repeated = lps[-1] > 0 and n % p == 0
```

**Checklist trước khi submit**
- [ ] Đã bỏ **cả** ký tự đầu và cuối của `s + s` chưa?
- [ ] Nếu thử ước: `L` chỉ tới `n // 2`, có kiểm tra `n % L == 0`?
- [ ] Ca `"a"` trả `False`, `"aa"` trả `True`?
- [ ] Nếu dùng KMP: có `lps[-1] > 0`?
- [ ] Giải thích được vì sao "xoay ra chính nó" ⇔ "lặp khối"?